# KBStats High-Average-Points Players

This notebook selects the newest timestamped KBStats player snapshot, requires a configurable minimum number of games, and retains players whose average points are at or above a configurable global percentile. The retained records are displayed and exported as timestamped JSON and CSV files.

The input timestamp is read from the filename rather than filesystem modification time. With the defaults, the percentile is calculated only among players with at least five games, and players tied at the cutoff are included.

## 1. Resolve project paths

In [1]:
# Import path utilities before locating the project workspace.
import sys
from pathlib import Path


# Handle project root for reuse in the workflow.
def _locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())

    checked = set()
    # Process each available item while preserving the current workflow state.
    for start in starts:
        # Process each available item while preserving the current workflow state.
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / "project_paths.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not locate project_paths.py. Start Jupyter from the Kickbase "
        "project root or open this notebook from within that project."
    )


# Set workflow configuration value: _PROJECT_ROOT.
_PROJECT_ROOT = _locate_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from project_paths import (
    DERIVED_KBSTATS_HIGH_AVERAGE_PLAYERS_DIR,
    KBSTATS_PLAYERS_DIR,
    ensure_directory,
)

## 2. Imports and configuration

Change `AVERAGE_POINTS_PERCENTILE` to select a different percentile. `MINIMUM_GAMES_PLAYED` controls the eligibility requirement.

In [2]:
# Import the libraries required by this notebook step.
from __future__ import annotations

import json
import math
import re
import warnings
from datetime import datetime, timezone
from typing import Any

import pandas as pd
from IPython.display import display

# Set workflow configuration value: AVERAGE_POINTS_PERCENTILE.
AVERAGE_POINTS_PERCENTILE = 75
# Set workflow configuration value: MINIMUM_GAMES_PLAYED.
MINIMUM_GAMES_PLAYED = 5

# Set workflow configuration value: KBSTATS_FILENAME_RE.
KBSTATS_FILENAME_RE = re.compile(
    r"^kbstats_players_"
    r"(?P<date>\d{8})_"
    r"(?P<time>\d{6})_"
    r"(?P<offset>[+-]\d{4})\.json$"
)

# Validate the input before continuing with later processing.
if (
    not isinstance(AVERAGE_POINTS_PERCENTILE, (int, float))
    or isinstance(AVERAGE_POINTS_PERCENTILE, bool)
    or not math.isfinite(float(AVERAGE_POINTS_PERCENTILE))
    or not 0 <= float(AVERAGE_POINTS_PERCENTILE) <= 100
):
    raise ValueError("AVERAGE_POINTS_PERCENTILE must be a finite number from 0 to 100.")
# Validate the input before continuing with later processing.
if (
    not isinstance(MINIMUM_GAMES_PLAYED, int)
    or isinstance(MINIMUM_GAMES_PLAYED, bool)
    or MINIMUM_GAMES_PLAYED < 1
):
    raise ValueError("MINIMUM_GAMES_PLAYED must be a positive integer.")

## 3. Select the newest KBStats snapshot

Candidate timestamps are parsed as timezone-aware instants. Malformed matching filenames are reported and ignored; two files encoding the same latest instant are treated as ambiguous.

In [3]:
# Parse and validate kbstats filename timestamp for reuse in the workflow.
def parse_kbstats_filename_timestamp(path: Path) -> datetime:
    match = KBSTATS_FILENAME_RE.fullmatch(path.name)
    # Validate the input before continuing with later processing.
    if match is None:
        raise ValueError(
            f"Filename does not contain a supported KBStats timestamp: {path.name}"
        )
    # Handle expected failures with a clear, actionable message.
    try:
        parsed = datetime.strptime(
            f"{match.group('date')}_{match.group('time')}_{match.group('offset')}",
            "%Y%m%d_%H%M%S_%z",
        )
    except ValueError as exc:
        raise ValueError(f"Invalid timestamp in {path.name}: {exc}") from exc
    return parsed.astimezone(timezone.utc)


# Select latest kbstats file for reuse in the workflow.
def select_latest_kbstats_file(directory: Path) -> Path:
    # Validate the input before continuing with later processing.
    if not directory.is_dir():
        raise FileNotFoundError(
            f"KBStats player output directory not found: {directory}. "
            "Run the KBStats player extraction notebook first."
        )

    candidates = sorted(directory.glob("kbstats_players_*.json"))
    # Validate the input before continuing with later processing.
    if not candidates:
        raise FileNotFoundError(
            f"No kbstats_players_*.json files were found in {directory}."
        )

    parsed_candidates: list[tuple[datetime, Path]] = []
    # Process each available item while preserving the current workflow state.
    for path in candidates:
        # Handle expected failures with a clear, actionable message.
        try:
            parsed_candidates.append((parse_kbstats_filename_timestamp(path), path))
        except ValueError as exc:
            warnings.warn(f"Ignoring {path.name}: {exc}", stacklevel=2)

    # Validate the input before continuing with later processing.
    if not parsed_candidates:
        raise ValueError(
            "KBStats JSON files were found, but none had a valid timezone-aware "
            "timestamp in the filename."
        )

    latest_timestamp = max(timestamp for timestamp, _ in parsed_candidates)
    latest_paths = [
        path for timestamp, path in parsed_candidates if timestamp == latest_timestamp
    ]
    # Validate the input before continuing with later processing.
    if len(latest_paths) != 1:
        names = ", ".join(path.name for path in latest_paths)
        raise RuntimeError(
            "Multiple KBStats files encode the same latest instant: " + names
        )
    return latest_paths[0]


selected_input_path = select_latest_kbstats_file(KBSTATS_PLAYERS_DIR)
print(f"Selected input: {selected_input_path}")

Selected input: C:\kickbase project\outputs\kbstats\players\kbstats_players_20260823_004913_+0200.json


## 4. Load and validate player records

Non-object records and records without finite numeric `gamesPlayed` and `averagePoints` values are excluded and counted. The original source records remain unchanged for JSON export.

In [4]:
# Check whether finite number for reuse in the workflow.
def is_finite_number(value: Any) -> bool:
    return (
        isinstance(value, (int, float))
        and not isinstance(value, bool)
        and math.isfinite(float(value))
    )


# Handle expected failures with a clear, actionable message.
try:
    raw_players = json.loads(selected_input_path.read_text(encoding="utf-8"))
except UnicodeDecodeError as exc:
    raise ValueError(f"Input is not valid UTF-8: {selected_input_path}") from exc
except json.JSONDecodeError as exc:
    raise ValueError(
        f"Invalid JSON at line {exc.lineno}, column {exc.colno}: {selected_input_path}"
    ) from exc
except OSError as exc:
    raise OSError(f"Could not read {selected_input_path}: {exc}") from exc

# Validate the input before continuing with later processing.
if not isinstance(raw_players, list):
    raise TypeError(
        f"Expected a JSON list of player records, got {type(raw_players).__name__}."
    )

analysis_rows: list[dict[str, Any]] = []
excluded_non_object_count = 0
excluded_invalid_numeric_count = 0

# Process each available item while preserving the current workflow state.
for source_index, player in enumerate(raw_players):
    if not isinstance(player, dict):
        excluded_non_object_count += 1
        continue
    games_played = player.get("gamesPlayed")
    average_points = player.get("averagePoints")
    if not is_finite_number(games_played) or not is_finite_number(average_points):
        excluded_invalid_numeric_count += 1
        continue
    player_name = player.get("name")
    if not isinstance(player_name, str) or not player_name.strip():
        player_name = " ".join(
            part.strip()
            for part in (player.get("firstName"), player.get("lastName"))
            if isinstance(part, str) and part.strip()
        )
    analysis_rows.append(
        {
            "source_index": source_index,
            "name": player_name,
            "name_sort": player_name.casefold(),
            "averagePoints": float(average_points),
            "gamesPlayed": float(games_played),
        }
    )

analysis_df = pd.DataFrame(analysis_rows)
# Validate the input before continuing with later processing.
if analysis_df.empty:
    raise ValueError("No player records contain usable gamesPlayed and averagePoints values.")

## 5. Apply the game minimum and percentile cutoff

The percentile is calculated only from eligible players. Pandas' linear quantile interpolation is used, and the comparison is inclusive so cutoff ties remain selected.

In [5]:
eligible_df = analysis_df.loc[
    analysis_df["gamesPlayed"] >= MINIMUM_GAMES_PLAYED
].copy()
# Validate the input before continuing with later processing.
if eligible_df.empty:
    raise ValueError(
        f"No players have at least {MINIMUM_GAMES_PLAYED} games with usable average points."
    )

average_points_cutoff = float(
    eligible_df["averagePoints"].quantile(
        float(AVERAGE_POINTS_PERCENTILE) / 100.0,
        interpolation="linear",
    )
)
selected_df = (
    eligible_df.loc[eligible_df["averagePoints"] >= average_points_cutoff]
    .sort_values(
        by=["averagePoints", "gamesPlayed", "name_sort"],
        ascending=[False, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)
selected_players = [
    raw_players[int(source_index)] for source_index in selected_df["source_index"]
]

display_rows = []
# Process each available item while preserving the current workflow state.
for player in selected_players:
    display_rows.append(
        {
            "player_id": player.get("id"),
            "name": player.get("name"),
            "team_id": player.get("teamId"),
            "position": player.get("position"),
            "average_points": player.get("averagePoints"),
            "games_played": player.get("gamesPlayed"),
            "total_points": player.get("totalPoints"),
            "market_value": player.get("marketValue"),
        }
    )

selected_display_df = pd.DataFrame(display_rows)
print("Processing summary")
print("------------------")
print(f"Source records: {len(raw_players):,}")
print(f"Excluded non-object records: {excluded_non_object_count:,}")
print(f"Excluded records with unusable numeric fields: {excluded_invalid_numeric_count:,}")
print(f"Eligible players (games >= {MINIMUM_GAMES_PLAYED}): {len(eligible_df):,}")
print(f"Average-points percentile: {float(AVERAGE_POINTS_PERCENTILE):g}")
print(f"Average-points cutoff: {average_points_cutoff:g}")
print(f"Selected players (ties included): {len(selected_players):,}")
display(selected_display_df)

Processing summary
------------------
Source records: 469
Excluded non-object records: 0
Excluded records with unusable numeric fields: 164
Eligible players (games >= 5): 285
Average-points percentile: 75
Average-points cutoff: 90
Selected players (ties included): 78


,player_id,name,team_id,position,average_points,games_played,total_points,market_value
0,8329,Michael Olise,2,3,225.0,32.0,7185.0,64765581.0
1,7226,Harry Kane,2,4,216.0,31.0,6703.0,68779146.0
2,1685,Joshua Kimmich,2,3,186.0,29.0,5391.0,59816839.0
3,11675,Luis Díaz,2,4,178.0,32.0,5698.0,53486340.0
4,2522,David Raum,43,2,148.0,30.0,4445.0,34053916.0
...,...,...,...,...,...,...,...,...
73,7723,Leopold Querfeld,40,2,90.0,31.0,2789.0,14383714.0
74,2887,Ritsu Dōan,4,3,90.0,31.0,2793.0,21489890.0
75,2854,Ermedin Demirović,9,4,90.0,25.0,2254.0,20614072.0
76,6300,Assan Ouédraogo,43,3,90.0,19.0,1705.0,13432757.0


## 6. Export matching JSON and CSV results

JSON preserves the selected source records. CSV flattens those records and serializes any remaining nested lists or dictionaries as JSON strings.

In [6]:
generated_datetime = datetime.now().astimezone()
generated_at = generated_datetime.isoformat(timespec="seconds")
output_timestamp = generated_datetime.strftime("%Y%m%d_%H%M%S_%z")
output_directory = ensure_directory(DERIVED_KBSTATS_HIGH_AVERAGE_PLAYERS_DIR)
json_output_path = output_directory / f"kbstats_high_average_players_{output_timestamp}.json"
csv_output_path = output_directory / f"kbstats_high_average_players_{output_timestamp}.csv"

output_document = {
    "generated_at": generated_at,
    "source_file": selected_input_path.name,
    "percentile": float(AVERAGE_POINTS_PERCENTILE),
    "average_points_cutoff": average_points_cutoff,
    "minimum_games_played": MINIMUM_GAMES_PLAYED,
    "eligible_player_count": len(eligible_df),
    "selected_player_count": len(selected_players),
    "players": selected_players,
}

# Handle expected failures with a clear, actionable message.
try:
    json_output_path.write_text(
        json.dumps(output_document, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
except OSError as exc:
    raise OSError(f"Could not write JSON output {json_output_path}: {exc}") from exc

csv_df = pd.json_normalize(selected_players, sep=".")
# Process each available item while preserving the current workflow state.
for column in csv_df.columns:
    csv_df[column] = csv_df[column].map(
        lambda value: (
            json.dumps(value, ensure_ascii=False)
            if isinstance(value, (list, dict))
            else value
        )
    )
# Handle expected failures with a clear, actionable message.
try:
    csv_df.to_csv(csv_output_path, index=False, encoding="utf-8-sig")
except OSError as exc:
    raise OSError(f"Could not write CSV output {csv_output_path}: {exc}") from exc

print(f"JSON output: {json_output_path}")
print(f"CSV output:  {csv_output_path}")

JSON output: C:\kickbase project\outputs\derived\kbstats_high_average_players\kbstats_high_average_players_20260823_005035_+0200.json
CSV output:  C:\kickbase project\outputs\derived\kbstats_high_average_players\kbstats_high_average_players_20260823_005035_+0200.csv


In [ ]:
from project_paths import prune_timestamped_outputs

removed_outputs = prune_timestamped_outputs()
print(f"Pruned {len(removed_outputs)} expired timestamped output(s).")
